# Notebook 53 — Deriving the Local Exponent from `Γ(x)`

This notebook extends Notebook 52.

Notebook 52 showed that if

`S(x) = exp(-∫₀ˣ Γ(u) du)`

then power-law effective-rate processes generate exact stretched exponentials.

This notebook derives the **local stretched exponent**

`b_local(x) = d log(-log S(x)) / d log x`

directly from `Γ(x)` and compares:

1. the exact exponent extracted from `S(x)`,
2. the exponent derived from the integral of `Γ(x)`,
3. the asymptotic exponent for power-law and corrected families.

## Main goals

- derive `b_local(x)` from first principles,
- prove that power-law `Γ(x)` gives constant local exponent,
- show how structured corrections deform the exponent,
- connect this to the fitted global exponent `b`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Core definitions

In [ ]:
x = np.linspace(1e-4, 1.0, 800)
dx = x[1] - x[0]

def build_S_from_gamma(gamma_x, x):
    integral = np.cumsum(gamma_x) * (x[1] - x[0])
    return np.exp(-integral), integral

def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 3.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse


## First-principles identity

Let

`I(x) = ∫₀ˣ Γ(u) du`

Then

`S(x) = exp(-I(x))`

and therefore

`-log S(x) = I(x)`

So the local stretched exponent is

`b_local(x) = d log I(x) / d log x`

Since

`dI/dx = Γ(x)`

we obtain the exact identity:

`b_local(x) = x Γ(x) / I(x)`

This notebook verifies that identity numerically and uses it as the central observable.


In [ ]:
def b_local_from_S(x, S):
    eps = 1e-14
    y = np.log(np.clip(-np.log(np.clip(S, eps, 1.0)), eps, None))
    z = np.log(np.clip(x, eps, None))
    return np.gradient(y, z)

def b_local_from_gamma(x, gamma_x, integral_x):
    eps = 1e-14
    return x * gamma_x / np.clip(integral_x, eps, None)


## Exact power-law families

In [ ]:
def gamma_power(x, c=1.8, m=0.9):
    return c * np.power(x, m - 1.0)

power_specs = [
    (1.5, 0.75),
    (1.8, 0.90),
    (2.0, 1.10),
]

rows_power = []
for c, m in power_specs:
    gamma_x = gamma_power(x, c=c, m=m)
    S, I = build_S_from_gamma(gamma_x, x)
    bS = b_local_from_S(x, S)
    bG = b_local_from_gamma(x, gamma_x, I)
    a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
    rows_power.append({
        "c": c,
        "m_true": m,
        "gamma": gamma_x,
        "S": S,
        "I": I,
        "b_from_S": bS,
        "b_from_gamma": bG,
        "b_fit": b_fit,
        "S_fit": S_fit,
        "fit_mse": mse,
    })

for row in rows_power:
    print(
        f'c={row["c"]:.2f}, m={row["m_true"]:.2f}, '
        f'b_fit={row["b_fit"]:.4f}, mse={row["fit_mse"]:.3e}'
    )


## Plot 1 — Exact local exponent identity for power-law rates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)

for row in rows_power:
    axes[0].plot(x, row["b_from_S"], label=f'm={row["m_true"]:.2f}')
    axes[1].plot(x, row["b_from_gamma"], label=f'm={row["m_true"]:.2f}')

axes[0].set_title("b_local(x) from S(x)")
axes[1].set_title("b_local(x) = x Γ(x) / ∫Γ")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("local exponent")
    ax.legend()

plt.tight_layout()
plt.show()


## Plot 2 — Compare exact local exponent to asymptotic m

In [ ]:
plt.figure(figsize=(8.2, 5.0))
for row in rows_power:
    plt.plot(x, row["b_from_gamma"], label=f'm={row["m_true"]:.2f}')
    plt.axhline(row["m_true"], linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("local exponent")
plt.title("For Γ(x)=c x^(m-1), the local exponent tends to m")
plt.legend()
plt.tight_layout()
plt.show()


## Structured corrections

Now take

`Γ(x) = c x^(m-1) [1 + ε h(x)]`

Then

`b_local(x) = x Γ(x) / ∫₀ˣ Γ(u) du`

is no longer constant.

This provides a first-principles explanation for why a single fitted `b`
is only an approximation when structured corrections are present.


In [ ]:
def gamma_corrected(x, c=1.8, m=0.9, eps=0.2, kind="linear"):
    base = c * np.power(x, m - 1.0)
    if kind == "linear":
        h = x
    elif kind == "quadratic":
        h = (x - 0.5)**2
    elif kind == "wave":
        h = np.sin(2*np.pi*x)
    else:
        h = np.zeros_like(x)
    return base * (1.0 + eps * h)

corr_specs = [
    ("linear", 0.20),
    ("quadratic", 0.30),
    ("wave", 0.15),
]

rows_corr = []
for kind, eps in corr_specs:
    gamma_x = gamma_corrected(x, c=1.8, m=0.9, eps=eps, kind=kind)
    S, I = build_S_from_gamma(gamma_x, x)
    bS = b_local_from_S(x, S)
    bG = b_local_from_gamma(x, gamma_x, I)
    a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
    rows_corr.append({
        "kind": kind,
        "eps": eps,
        "gamma": gamma_x,
        "S": S,
        "I": I,
        "b_from_S": bS,
        "b_from_gamma": bG,
        "b_fit": b_fit,
        "S_fit": S_fit,
        "fit_mse": mse,
    })

for row in rows_corr:
    print(
        f'{row["kind"]}: eps={row["eps"]:.2f}, '
        f'b_fit={row["b_fit"]:.4f}, mse={row["fit_mse"]:.3e}'
    )


## Plot 3 — Corrected local exponents are scale-dependent

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)

for row in rows_corr:
    axes[0].plot(x, row["b_from_S"], label=f'{row["kind"]}')
    axes[1].plot(x, row["b_from_gamma"], label=f'{row["kind"]}')

axes[0].set_title("b_local(x) from S(x)")
axes[1].set_title("b_local(x) from Γ(x)")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("local exponent")
    ax.legend()

plt.tight_layout()
plt.show()


## Plot 4 — Global fitted b vs local exponent profile

In [ ]:
plt.figure(figsize=(8.3, 5.1))
for row in rows_corr:
    plt.plot(x, row["b_from_gamma"], label=f'{row["kind"]} local')
    plt.axhline(row["b_fit"], linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("exponent")
plt.title("Global fitted b is a coarse summary of b_local(x)")
plt.legend()
plt.tight_layout()
plt.show()


## Deriving the first correction

For the corrected family

`Γ(x) = c x^(m-1) [1 + ε h(x)]`

we have

`I(x) = (c/m) x^m + ε c ∫₀ˣ u^(m-1) h(u) du`

and therefore

`b_local(x) = x Γ(x) / I(x)`

Expanding to first order in `ε`, the local exponent is shifted by the shape function `h`
through both the numerator and the integral in the denominator.

This is why slope, curvature, and range of `Γ(x)` perturb the global fitted exponent.


In [ ]:
def summarize_local_profile(x, b_local):
    return {
        "mean_local_b": float(np.mean(b_local)),
        "std_local_b": float(np.std(b_local)),
        "range_local_b": float(np.max(b_local) - np.min(b_local)),
    }

summary_rows = []
for row in rows_corr:
    s = summarize_local_profile(x, row["b_from_gamma"])
    summary_rows.append({
        "kind": row["kind"],
        "b_fit": row["b_fit"],
        **s
    })

for row in summary_rows:
    print(row)


## Plot 5 — Fitted global exponent tracks the local-exponent profile

In [ ]:
mean_local = np.array([r["mean_local_b"] for r in summary_rows], dtype=float)
range_local = np.array([r["range_local_b"] for r in summary_rows], dtype=float)
b_fit_vals = np.array([r["b_fit"] for r in summary_rows], dtype=float)
names = [r["kind"] for r in summary_rows]

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))

axes[0].scatter(mean_local, b_fit_vals, s=85)
for i, name in enumerate(names):
    axes[0].annotate(name, (mean_local[i], b_fit_vals[i]), fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[0].set_xlabel("mean local exponent")
axes[0].set_ylabel("global fitted b")
axes[0].set_title("Global b vs mean local exponent")

axes[1].scatter(range_local, b_fit_vals, s=85)
for i, name in enumerate(names):
    axes[1].annotate(name, (range_local[i], b_fit_vals[i]), fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[1].set_xlabel("range of local exponent")
axes[1].set_ylabel("global fitted b")
axes[1].set_title("Global b vs variability of b_local(x)")

plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Deriving the local exponent from Γ(x)")
lines.append("")
lines.append("Exact identity:")
lines.append("b_local(x) = x Γ(x) / ∫₀ˣ Γ(u) du")
lines.append("")
for row in rows_power:
    lines.append(
        f'Power-law family m={row["m_true"]:.2f}: global fit b={row["b_fit"]:.4f}, fit_MSE={row["fit_mse"]:.3e}'
    )
lines.append("")
for row in rows_corr:
    lines.append(
        f'Corrected {row["kind"]}: global fit b={row["b_fit"]:.4f}, fit_MSE={row["fit_mse"]:.3e}'
    )
lines.append("")
lines.append("Interpretation:")
lines.append("- For Γ(x)=c x^(m-1), the local exponent is constant and equals m.")
lines.append("- Structured corrections make b_local(x) scale-dependent.")
lines.append("- The fitted global b is therefore a coarse average of a local exponent profile.")
print("\n".join(lines))


## Final conclusion

This notebook derives the local stretched exponent directly from the effective-rate process.

The key exact identity is:

`b_local(x) = x Γ(x) / ∫₀ˣ Γ(u) du`

Consequences:

1. if `Γ(x)` is a power law, the exponent is constant and stretched-exponential behavior is exact,
2. if `Γ(x)` has structured corrections, the exponent becomes scale-dependent,
3. the fitted global `b` used in earlier notebooks is a coarse summary of this local profile.

This completes the first-principles picture:

`Γ(x)`  
→ local exponent `b_local(x)`  
→ global fitted exponent `b`
